In [1]:
# =============================================================================
#  Fermi Velocity Prediction Pipeline  v2  — family + subfamily features
#  Input  : train_raw.csv, unique_m.csv, fermi_759.csv
#  Output : UCI_with_Fermi_Velocity.csv
#           (21,263 rows | 81 features + material + family + subfamily + Predicted_vF)
# =============================================================================

import pandas as pd
import numpy as np
import re
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# STEP 1 — Load data
# =============================================================================
df_train    = pd.read_csv("train_raw.csv")
df_unique_m = pd.read_csv("unique_m.csv")
# df_fermi    = pd.read_csv("fermi_759.csv")   # 759 rows | 169 unique materials
df_fermi    = pd.read_csv("Fermi_169_unique_v5.csv")   # 759 rows | 169 unique materials

# Keep ORIGINAL case for regex-based classification; lowercase only as merge key
df_unique_m['material_lower'] = df_unique_m['material'].str.strip().str.lower()
df_unique_m['material_orig']  = df_unique_m['material'].str.strip()
df_fermi['material']          = df_fermi['material'].str.strip().str.lower()

# lookup: lowercase → original case
orig_case_map = dict(zip(df_unique_m['material_lower'], df_unique_m['material_orig']))


In [ ]:
## Old version
# =============================================================================
# STEP 2 — Classification functions (operate on ORIGINAL case strings)
# =============================================================================
def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter → avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        # FIX-8: like has(), but requires the element's stoichiometric
        # coefficient to be >= threshold. No explicit number = coefficient 1.
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'   
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
       # FIX-1: MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'
 
    # FIX-2: heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'
 
    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        if has('Ce', mat) and has('In', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']):
            return 'cuprate_LSCO'
        if has('Bi', mat) and has('Sr', mat):
            return 'cuprate_BSCCO'
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        if has('Ba', mat) and (has('Y', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # FIX-4: Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
        # IRON-BASED
    if has_major('Fe', mat):
        if has('As', mat) and has('O', mat):
            return 'iron_1111'
        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # FIX-5: 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'
 
    # ── HEAVY FERMION ─────────────────────────────────────────────────────
        # HEAVY FERMION
    # FIX-3: Cu/Si added
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'
   
    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    # BOROCARBIDE
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'
    
    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
        # CONVENTIONAL A15
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga']) and not has_major('Fe', mat):
        return 'conventional_A15'  
    
    # ── CONVENTIONAL NbTi / transition metals ─────────────────────────────
    if has('Nb', mat):
        return 'conventional_Nb_based'     
    
    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────    
    # FIX-6: CHALCOGENIDE before SIMPLE METALS
    # Sn0.97As0.022Te0.978, PbMo6S8 hit chalcogenide before Sn/Pb fires simple_metal
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    # SIMPLE METALS
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'
 
    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':         'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# FIX-9: Flag low-confidence subfamily assignments
#
# conventional_Nb_based in the 169-sample training set is really a catch-all for
# simple Nb-binary/ternary compounds (alloys, borides, carbides, simple oxides
# like LiNbO2, NbOS2). The 21,263-row UCI set also contains complex layered
# Ruddlesden-Popper niobates (e.g. KCa2Nb3O10, LiRbCa2Nb3O10 -- 4+ elements,
# Nb + O, no Ti). These have no analog in the training set, so predictions for
# them are extrapolation. Flag them rather than silently trusting the vF lookup.
# =============================================================================
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

# FIX-7: Single source of truth for vF — replaces raw CSV vF column
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          850000,
    'conventional_NbTi':         760000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}
 
def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)
 


In [9]:
## New version v1
import re

def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.

    CHANGE LOG (relative to fermi_velocity_169_fixed_1.ipynb):
      1. NEW: alkali-intercalated (K/Rb/Cs/Tl) FeSe-derived compounds
         (e.g. K0.8Fe1.7Se2) now route to 'iron_245' instead of being
         silently absorbed into 'iron_11' (simple binary FeSe/FeTe).
         Inserted BEFORE the iron_11 check.
      2. Ca-Pt-intercalated Fe-As "10-3-8"/"10-4-8" phase compounds
         (e.g. Ca10Pt3As8(Fe2As2)5) now route to 'iron_other' instead of
         being silently absorbed into 'iron_122' (standard ThCr2Si2-type).
         No dedicated subfamily is created, since no literature vF value
         exists for this class either way -- both 'iron_1038' and
         'iron_other' are routed through the same ML fallback path, so a
         separate label adds no behavioral difference. Inserted BEFORE
         the iron_122 check.
      3. A15 element check extended with 'Sn' AND 'In' so V3Sn- and
         V3In-type stoichiometry (V + Sn or In, no Si/Ga needed) routes
         to 'conventional_A15' instead of falling through to
         'conventional_simple_metal'.
      4. FIX: cuprate_LSCO check now excludes compounds where Ba is
         present at MAJOR stoichiometry (>=0.5) TOGETHER WITH a rare
         earth from the YBCO RE list (Er, Gd, Sm, Nd, Ho, Dy, Eu, Lu, Pr,
         Tb, Tm, Yb) -- not just Y/Bi/Tl/Hg, and not RE-presence or
         trace-Ba presence alone. Previously, a genuine YBCO-type 123
         structure such as La0.5Er0.5Ba2Cu3O was caught by the La-only
         LSCO check before ever reaching the YBCO check below it, since
         the LSCO exclusion list did not include Er. The major-Ba
         threshold (matching the real ~2-per-formula-unit Ba2Cu3 block
         that defines 123-type structures) is required in addition to RE
         presence, because compounds such as La1.6-xNd0.4SrxCuO4
         ("Nd-LSCO") are a well-established, named branch of the 214-type
         LSCO family in the literature -- Nd doping on the La site, even
         alongside trace Ba doping, does not indicate a 123-type
         structure, and an RE-alone or any-Ba exclusion would have
         wrongly pulled these genuine LSCO compounds out of cuprate_LSCO.
      5. FIX: Bi+Sr (BSCCO) check now excludes compounds that also
         contain Hg. Previously, any compound containing both Bi and Hg
         (e.g. Hg0.5Bi0.5Sr2Ca2Cu3O8.35, a genuine mixed/hybrid
         composition) was unconditionally routed to cuprate_BSCCO, since
         the BSCCO check ran before the Hg-based check with no tie-break.
         Such hybrid compositions now route to cuprate_other rather than
         being silently assigned to a single parent structure.
      6. FIX: the nested Ce+In check inside the Cu+O cuprate block (used
         to catch Ce/In-based heavy-fermion superconductors that also
         happen to satisfy has('Cu') and has('O')) now carries the same
         not has('O') guard as the standalone heavy_fermion_115 check
         above it. Previously this nested check had no O guard, so any
         genuine Ce+In co-doped cuprate (e.g. an electron-doped compound
         such as Nd1.88Ce0.12Cu0.98In0.02O4) was misrouted to
         heavy_fermion_115 instead of cuprate_electron_doped.

    This function is unchanged in every other respect, including remaining
    fully case-sensitive as originally written. It must be called on
    ORIGINAL-CASE material strings (material_orig / orig_case_map), not on
    the lowercase 'material' column -- calling it on lowercase input will
    silently fail to match any element and return 'other' for everything.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        # FIX (CHANGE 6): added not has('O', mat) guard, matching the
        # standalone heavy_fermion_115 check above. Without this guard,
        # any Ce+In co-doped cuprate (which always has O, by construction
        # of this block) was misrouted here instead of falling through to
        # the electron-doped cuprate check below.
        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        # FIX (CHANGE 4, updated): LSCO check now excludes compounds where
        # Ba is present at MAJOR stoichiometry (>=0.5, the real 123-type
        # Ba2Cu3-block signature) regardless of whether a second rare
        # earth is present. Literature (Ganguly & Sreedhar review of
        # RBa2Cu3O-delta phases) confirms pure LaBa2Cu3O-delta -- with La
        # as the ONLY rare earth, substituting partially onto the Ba site
        # -- is a genuine, documented member of the 123/YBCO family, not
        # a separate structure. An earlier version of this fix required a
        # SECOND rare earth (e.g. Er) alongside Ba, which correctly
        # excluded true 214-type "Nd-LSCO" compounds (La+Nd+Sr, only
        # trace Ba) but incorrectly still routed pure La+Ba2+Cu3+O7-type
        # compounds (no second RE) to cuprate_LSCO. The condition now
        # keys on major-Ba alone: trace-Ba Nd-LSCO compounds (Ba~0.14)
        # still correctly stay LSCO, since 0.14 < 0.5, while major-Ba
        # La-123 compounds (Ba~1.3-2) now correctly route to cuprate_YBCO.
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        # FIX (CHANGE 5): BSCCO check now excludes compounds that also
        # contain Hg, since Bi+Hg together indicates a genuine hybrid
        # composition rather than a clean BSCCO structure. Such compounds
        # now fall through to cuprate_other instead of being silently
        # assigned to BSCCO ahead of the Hg-based check below.
        if has('Bi', mat) and has('Sr', mat) and not has('Hg', mat):
            return 'cuprate_BSCCO'
        # FIX (CHANGE 8 continued): YBCO check's rare-earth trigger now
        # also includes La. Previously only Y and the RE list (Er, Gd,
        # Sm, Nd, Ho, Dy, Eu, Lu, Pr, Tb, Tm, Yb) reached this check --
        # La was excluded here too (it is handled as its own special
        # case throughout this function). Combined with the LSCO-check
        # fix above, a pure La+Ba(major)+Cu3+O7-type compound (e.g.
        # La1.5Ba1.3Ca0.2Cu3O6.897, no second rare earth) would otherwise
        # fall through BOTH the LSCO check (correctly excluded) and this
        # YBCO check (La not recognized as a trigger), landing in
        # cuprate_other instead of the correct cuprate_YBCO.
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        # NEW (CHANGE 2): Ca-Pt-arsenide "10-3-8"/"10-4-8" phase.
        # Requires Ca AND Pt together (the defining Ca10(Pt3As8)(Fe2As2)5
        # signature). Routed to iron_other (same ML-fallback path as
        # iron_1038 would have used) rather than a separate named
        # subfamily, since no literature vF value exists for this class.
        # Must come BEFORE the iron_1111 and iron_122 checks.
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        # FIX (CHANGE 8): 'or has('H', mat)' added. Literature (Hanna et
        # al., PRB 89, 094501; "Phase transition in CaFeAsH: bridging
        # 1111 and 122 iron-based superconductors," Dalton Trans. 2018;
        # and the La-substituted CaFeAsH Tc=47K report) confirms hydride
        # 1111-type iron arsenides (CaFeAsH, LnFeAsO1-xHx) adopt the same
        # ZrCuSiAs-type structure as the oxide 1111 family, with H- (a
        # hydride anion, isovalent with F-) substituting directly for O2-
        # in the "blocking layer." H is therefore structurally equivalent
        # to O for this check, not a separate signal. Without this, a
        # compound like Ca0.8La0.2FeAsH (no O at all) failed the O-only
        # check and fell through to iron_122 instead.
        if has('As', mat) and (has('O', mat) or has('H', mat)):
            return 'iron_1111'

        # NEW (CHANGE 1): alkali-intercalated 245-type selenide.
        # Must come BEFORE the plain iron_11 check. Na added alongside
        # K/Rb/Cs/Tl -- literature (Guo et al., Nat. Commun. 2015; Fang
        # et al., Sci. Rep. 2012) documents NaxFe2-ySe2 as the same
        # ThCr2Si2-type "122-like" alkali-intercalated structural family
        # as K/Rb/Cs-Fe-Se, distinct from simple binary FeSe/FeTe (iron_11).
        if any(has(a, mat) for a in ['K', 'Rb', 'Cs', 'Tl', 'Na']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    # MODIFIED (CHANGE 3): added 'Sn', 'In', and 'Al'. V-Al alloys are
    # explicitly documented A15-structure superconductors (see e.g.
    # Zeitschrift fur Physik B, "Critical temperature ... of superconducting
    # V-Ga-Al alloys with A15 structure").
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    # Was 'conventional_NbTi'. Renamed to 'conventional_Nb_based' since the
    # actual contents of this bucket are general Nb binary/ternary
    # compounds (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2),
    # not specifically Nb-Ti alloys -- the old name implied a narrower,
    # inaccurate structural claim.
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    # BaBiO3-derived perovskite/CDW superconductors: BaPb1-xBixO3 (BPBO) and
    # Ba1-xKxBiO3 (BKBO). Structurally and mechanistically distinct from the
    # sp-band alloys in conventional_simple_metal (non-magnetic CDW-derived
    # perovskite oxide, not a simple metallic alloy) -- see e.g. Nature
    # Materials/Nature Communications reviews of the BPBO/BKBO family.
    # Must come BEFORE the simple-metals check, since Pb/Al/In there would
    # otherwise catch these compounds first.
    # No literature vF value is assigned for this class -- routed to
    # 'other' (ML fallback), matching how iron_1038 and other
    # structurally-distinct-but-unvalued classes are handled.
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')

# =============================================================================
# FIX-9: Flag low-confidence subfamily assignments
#
# conventional_Nb_based in the 169-sample training set is really a catch-all for
# simple Nb-binary/ternary compounds (alloys, borides, carbides, simple oxides
# like LiNbO2, NbOS2). The 21,263-row UCI set also contains complex layered
# Ruddlesden-Popper niobates (e.g. KCa2Nb3O10, LiRbCa2Nb3O10 -- 4+ elements,
# Nb + O, no Ti). These have no analog in the training set, so predictions for
# them are extrapolation. Flag them rather than silently trusting the vF lookup.
# =============================================================================
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

# =============================================================================
# FIX-7 (unchanged) + NEW iron_245 entry -- Single source of truth for vF
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_245':                   79000,   # NEW -- Qian et al., PRL 106, 187001 (2011)
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          212100,
    'conventional_Nb_based':     726000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

In [ ]:
## New version v2
import re

def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.

    CHANGE LOG (relative to fermi_velocity_169_fixed_1.ipynb):
      1. NEW: alkali-intercalated (K/Rb/Cs/Tl) FeSe-derived compounds
         (e.g. K0.8Fe1.7Se2) now route to 'iron_245' instead of being
         silently absorbed into 'iron_11' (simple binary FeSe/FeTe).
         Inserted BEFORE the iron_11 check.
      2. Ca-Pt-intercalated Fe-As "10-3-8"/"10-4-8" phase compounds
         (e.g. Ca10Pt3As8(Fe2As2)5) now route to 'iron_other' instead of
         being silently absorbed into 'iron_122' (standard ThCr2Si2-type).
         No dedicated subfamily is created, since no literature vF value
         exists for this class either way -- both 'iron_1038' and
         'iron_other' are routed through the same ML fallback path, so a
         separate label adds no behavioral difference. Inserted BEFORE
         the iron_122 check.
      3. A15 element check extended with 'Sn' AND 'In' so V3Sn- and
         V3In-type stoichiometry (V + Sn or In, no Si/Ga needed) routes
         to 'conventional_A15' instead of falling through to
         'conventional_simple_metal'.
      4. FIX: cuprate_LSCO check now excludes compounds where Ba is
         present at MAJOR stoichiometry (>=0.5) TOGETHER WITH a rare
         earth from the YBCO RE list (Er, Gd, Sm, Nd, Ho, Dy, Eu, Lu, Pr,
         Tb, Tm, Yb) -- not just Y/Bi/Tl/Hg, and not RE-presence or
         trace-Ba presence alone. Previously, a genuine YBCO-type 123
         structure such as La0.5Er0.5Ba2Cu3O was caught by the La-only
         LSCO check before ever reaching the YBCO check below it, since
         the LSCO exclusion list did not include Er. The major-Ba
         threshold (matching the real ~2-per-formula-unit Ba2Cu3 block
         that defines 123-type structures) is required in addition to RE
         presence, because compounds such as La1.6-xNd0.4SrxCuO4
         ("Nd-LSCO") are a well-established, named branch of the 214-type
         LSCO family in the literature -- Nd doping on the La site, even
         alongside trace Ba doping, does not indicate a 123-type
         structure, and an RE-alone or any-Ba exclusion would have
         wrongly pulled these genuine LSCO compounds out of cuprate_LSCO.
      5. FIX: Bi+Sr (BSCCO) check now excludes compounds that also
         contain Hg. Previously, any compound containing both Bi and Hg
         (e.g. Hg0.5Bi0.5Sr2Ca2Cu3O8.35, a genuine mixed/hybrid
         composition) was unconditionally routed to cuprate_BSCCO, since
         the BSCCO check ran before the Hg-based check with no tie-break.
         Such hybrid compositions now route to cuprate_other rather than
         being silently assigned to a single parent structure.
      6. FIX: the nested Ce+In check inside the Cu+O cuprate block (used
         to catch Ce/In-based heavy-fermion superconductors that also
         happen to satisfy has('Cu') and has('O')) now carries the same
         not has('O') guard as the standalone heavy_fermion_115 check
         above it. Previously this nested check had no O guard, so any
         genuine Ce+In co-doped cuprate (e.g. an electron-doped compound
         such as Nd1.88Ce0.12Cu0.98In0.02O4) was misrouted to
         heavy_fermion_115 instead of cuprate_electron_doped.

    This function is unchanged in every other respect, including remaining
    fully case-sensitive as originally written. It must be called on
    ORIGINAL-CASE material strings (material_orig / orig_case_map), not on
    the lowercase 'material' column -- calling it on lowercase input will
    silently fail to match any element and return 'other' for everything.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    def has_explicit_coefficient_ge(element, string, threshold):
        # Stricter than has_major(): requires an EXPLICIT numeric
        # coefficient >= threshold. A bare element with no written number
        # (e.g. 'Se' in 'FeSe', implicit coefficient 1) does NOT
        # automatically pass here, unlike has_major(). Needed for
        # thresholds well above 1, where "no number written" must not be
        # treated as satisfying the threshold (see CHANGE 10 below).
        m = re.search(rf"{element}(?![a-z])([0-9]+\.?[0-9]*)", string)
        if m is None:
            return False
        return float(m.group(1)) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        # FIX (CHANGE 6): added not has('O', mat) guard, matching the
        # standalone heavy_fermion_115 check above. Without this guard,
        # any Ce+In co-doped cuprate (which always has O, by construction
        # of this block) was misrouted here instead of falling through to
        # the electron-doped cuprate check below.
        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        # FIX (CHANGE 4, updated): LSCO check now excludes compounds where
        # Ba is present at MAJOR stoichiometry (>=0.5, the real 123-type
        # Ba2Cu3-block signature) regardless of whether a second rare
        # earth is present. Literature (Ganguly & Sreedhar review of
        # RBa2Cu3O-delta phases) confirms pure LaBa2Cu3O-delta -- with La
        # as the ONLY rare earth, substituting partially onto the Ba site
        # -- is a genuine, documented member of the 123/YBCO family, not
        # a separate structure. An earlier version of this fix required a
        # SECOND rare earth (e.g. Er) alongside Ba, which correctly
        # excluded true 214-type "Nd-LSCO" compounds (La+Nd+Sr, only
        # trace Ba) but incorrectly still routed pure La+Ba2+Cu3+O7-type
        # compounds (no second RE) to cuprate_LSCO. The condition now
        # keys on major-Ba alone: trace-Ba Nd-LSCO compounds (Ba~0.14)
        # still correctly stay LSCO, since 0.14 < 0.5, while major-Ba
        # La-123 compounds (Ba~1.3-2) now correctly route to cuprate_YBCO.
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        # FIX (CHANGE 5): BSCCO check now excludes compounds that also
        # contain Hg, since Bi+Hg together indicates a genuine hybrid
        # composition rather than a clean BSCCO structure. Such compounds
        # now fall through to cuprate_other instead of being silently
        # assigned to BSCCO ahead of the Hg-based check below.
        if has('Bi', mat) and has('Sr', mat) and not has('Hg', mat):
            return 'cuprate_BSCCO'
        # FIX (CHANGE 8 continued): YBCO check's rare-earth trigger now
        # also includes La. Previously only Y and the RE list (Er, Gd,
        # Sm, Nd, Ho, Dy, Eu, Lu, Pr, Tb, Tm, Yb) reached this check --
        # La was excluded here too (it is handled as its own special
        # case throughout this function). Combined with the LSCO-check
        # fix above, a pure La+Ba(major)+Cu3+O7-type compound (e.g.
        # La1.5Ba1.3Ca0.2Cu3O6.897, no second rare earth) would otherwise
        # fall through BOTH the LSCO check (correctly excluded) and this
        # YBCO check (La not recognized as a trigger), landing in
        # cuprate_other instead of the correct cuprate_YBCO.
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        # NEW (CHANGE 2): Ca-Pt-arsenide "10-3-8"/"10-4-8" phase.
        # Requires Ca AND Pt together (the defining Ca10(Pt3As8)(Fe2As2)5
        # signature). Routed to iron_other (same ML-fallback path as
        # iron_1038 would have used) rather than a separate named
        # subfamily, since no literature vF value exists for this class.
        # Must come BEFORE the iron_1111 and iron_122 checks.
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        # FIX (CHANGE 8): 'or has('H', mat)' added. Literature (Hanna et
        # al., PRB 89, 094501; "Phase transition in CaFeAsH: bridging
        # 1111 and 122 iron-based superconductors," Dalton Trans. 2018;
        # and the La-substituted CaFeAsH Tc=47K report) confirms hydride
        # 1111-type iron arsenides (CaFeAsH, LnFeAsO1-xHx) adopt the same
        # ZrCuSiAs-type structure as the oxide 1111 family, with H- (a
        # hydride anion, isovalent with F-) substituting directly for O2-
        # in the "blocking layer." H is therefore structurally equivalent
        # to O for this check, not a separate signal. Without this, a
        # compound like Ca0.8La0.2FeAsH (no O at all) failed the O-only
        # check and fell through to iron_122 instead.
        if has('As', mat) and (has('O', mat) or has('H', mat)):
            return 'iron_1111'

        # NEW (CHANGE 10): quasi-1D "ladder" AFe2Se3 phase (A = Ba, K, Cs).
        # Must come BEFORE both the iron_245 and iron_11 checks below, since
        # these compounds would otherwise be caught by either (they contain
        # the same alkali/alkaline-earth intercalants as iron_245, and no
        # As, same as iron_11). Literature (Krzton-Maziopa et al., J. Phys.
        # Condens. Matter 23, 402201, 2011; Caron et al., PRB 84, 180409,
        # 2011; Nature Materials 14, 745, 2015) confirms AFe2Se3 is a
        # structurally DISTINCT family from both: built from quasi-1D
        # two-leg Fe-Se ladders (edge-sharing FeSe4 tetrahedra), not the 2D
        # FeSe layers found in either iron_11 (simple binary FeSe/FeTe) or
        # iron_245 (2D AFe2Se2 intercalated layers). Detected here by
        # Fe:Se stoichiometry near 2:3 (vs 1:1 for iron_11, 1:2 for
        # iron_245). No literature vF exists for this niche, primarily
        # magnetically-studied family -- routed to iron_other (ML fallback)
        # rather than a hand-picked or borrowed lookup value.
        if has_explicit_coefficient_ge('Se', mat, 2.5) and not has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 1, extended): alkali- OR alkaline-earth-intercalated
        # 245-type selenide. Ca/Sr/Ba/Eu/Yb added alongside the original
        # K/Rb/Cs/Tl/Na alkali list -- literature confirms AFe2Se2 (A = Ca,
        # Sr, Ba, Eu, Yb, in addition to the alkalis) shares the same
        # ThCr2Si2-type "122-like" intercalated structure, distinct from
        # simple binary FeSe/FeTe (iron_11). Detected by Fe:Se ~= 1:2
        # stoichiometry (the ladder-compound check above, which requires
        # Se >= 2.5, already excludes the 2:3-stoichiometry AFe2Se3 ladder
        # compounds from reaching this branch).
        if any(has(a, mat) for a in ['Li', 'K', 'Rb', 'Cs', 'Tl', 'Na', 'Ca', 'Sr', 'Ba', 'Eu', 'Yb']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    # MODIFIED (CHANGE 3): added 'Sn', 'In', and 'Al'. V-Al alloys are
    # explicitly documented A15-structure superconductors (see e.g.
    # Zeitschrift fur Physik B, "Critical temperature ... of superconducting
    # V-Ga-Al alloys with A15 structure").
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    # Was 'conventional_NbTi'. Renamed to 'conventional_Nb_based' since the
    # actual contents of this bucket are general Nb binary/ternary
    # compounds (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2),
    # not specifically Nb-Ti alloys -- the old name implied a narrower,
    # inaccurate structural claim.
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    # BaBiO3-derived perovskite/CDW superconductors: BaPb1-xBixO3 (BPBO) and
    # Ba1-xKxBiO3 (BKBO). Structurally and mechanistically distinct from the
    # sp-band alloys in conventional_simple_metal (non-magnetic CDW-derived
    # perovskite oxide, not a simple metallic alloy) -- see e.g. Nature
    # Materials/Nature Communications reviews of the BPBO/BKBO family.
    # Must come BEFORE the simple-metals check, since Pb/Al/In there would
    # otherwise catch these compounds first.
    # No literature vF value is assigned for this class -- routed to
    # 'other' (ML fallback), matching how iron_1038 and other
    # structurally-distinct-but-unvalued classes are handled.
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# vF LOOKUP TABLE -- direct-lookup subfamilies only.
#
# IMPORTANT: 'other', 'iron_other', and 'cuprate_other' are INTENTIONALLY
# ABSENT from this dictionary. These are the catch-all buckets for
# structurally ambiguous or genuinely mixed compounds (e.g. the Ca-Pt-As
# "10-3-8" phase inside iron_other, Hg+Bi hybrid cuprates inside
# cuprate_other, and chemically unrelated compounds like bismuthate
# perovskites or Fe-Ni-Zr Laves phases inside the generic 'other'). No
# single literature-backed Fermi velocity exists for these buckets, since
# they are deliberately heterogeneous by construction, not a real physical
# family. Per the actual training pipeline (vF_xi_Hc2.ipynb, Section 2:
# AMBIGUOUS_SUBFAMILIES / is_ambiguous_row), rows in these three subfamilies
# are ALWAYS routed to the trained ML fallback model (final_model.predict,
# followed by snap_to_valid) instead of a fixed lookup value. An earlier
# version of this file incorrectly gave these three subfamilies a borrowed
# or arbitrary constant (cuprate_other = cuprate_YBCO's value, iron_other =
# iron_122's value, other = an unexplained 760,000 default) -- these were
# never real citations and did not match how the actual pipeline behaves,
# so get_vF() below now raises rather than silently returning a number that
# looks citation-backed but isn't.
#
# conventional_A15 updated to the literature value: Mentink et al.,
# Supercond. Sci. Technol. 30, 025006 (2017) -- ab initio DFT calculation
# of Nb3Sn, RMS Fermi velocity vF = 2.121e5 m/s, cross-validated against
# four independent experimental sources (Lim, Schachinger, Orlando, Ghosh)
# in the same paper. Replaces the previous unsourced 850,000 m/s.
#
# conventional_Nb_based updated to: Prozorov, Zarea, and Sauls, arXiv:
# 2207.11829 (2022), citing the underlying DFT calculation of elemental Nb
# by Zarea, Ueki, and Sauls, arXiv:2201.07403 (2022). Two-band-averaged
# RMS Fermi velocity v = 7.26e5 m/s. This is elemental niobium's own value,
# used as the representative constant for the whole conventional_Nb_based
# bucket (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2, etc.)
# since Nb is the one element common to every compound in this bucket;
# compound-specific corrections (carbide vs boride vs oxide bonding) have
# no citable literature basis and are not applied.
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  45600,
    'iron_11':                    13700,
    'iron_245':                   79000,   # Qian et al., PRL 106, 187001 (2011)
    'MgB2_type':                 560000,
    'conventional_A15':          212100,   # Mentink et al., SST 30, 025006 (2017)
    'conventional_Nb_based':     726000,   # Prozorov, Zarea & Sauls, arXiv:2207.11829 (2022)
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    # 'cuprate_other', 'iron_other', 'other' deliberately absent -- see above.
}

# Subfamilies with no literature-backed vF, routed to the trained ML
# fallback model instead of VF_LOOKUP. Matches AMBIGUOUS_SUBFAMILIES in
# vF_xi_Hc2.ipynb exactly.
# AMBIGUOUS_SUBFAMILIES = {"other", "iron_other", "cuprate_other"}

# def is_ambiguous_row(subfamily, low_confidence=False):
#     """True if this row must go through the ML fallback instead of direct lookup."""
#     return (subfamily in AMBIGUOUS_SUBFAMILIES) or bool(low_confidence)

# def get_vF(mat_orig, low_confidence=False):
#     """
#     Direct-lookup Fermi velocity for non-ambiguous subfamilies only.

#     Raises ValueError for 'other', 'iron_other', 'cuprate_other', or any
#     row flagged low_confidence, since these have no literature-backed
#     lookup value by design -- they must be scored by the trained ML
#     fallback model instead (see vF_xi_Hc2.ipynb, Section 2 and Section 10).
#     This function intentionally does NOT silently return a placeholder for
#     these cases, since an earlier version's silent fallback (a borrowed or
#     arbitrary constant) is not something you can defend to a reviewer as a
#     physical value.
#     """
#     sf = get_subfamily(mat_orig)
#     if is_ambiguous_row(sf, low_confidence):
#         raise ValueError(
#             f"'{sf}' has no literature-backed vF lookup value -- route this "
#             f"compound through the trained ML fallback model instead."
#         )
#     return VF_LOOKUP[sf]
# =============================================================================
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

In [6]:
## New version v3
import re

def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.

    CHANGE LOG (relative to fermi_velocity_169_fixed_1.ipynb):
      1. NEW: alkali-intercalated (K/Rb/Cs/Tl) FeSe-derived compounds
         (e.g. K0.8Fe1.7Se2) now route to 'iron_245' instead of being
         silently absorbed into 'iron_11' (simple binary FeSe/FeTe).
         Inserted BEFORE the iron_11 check.
      2. Ca-Pt-intercalated Fe-As "10-3-8"/"10-4-8" phase compounds
         (e.g. Ca10Pt3As8(Fe2As2)5) now route to 'iron_other' instead of
         being silently absorbed into 'iron_122' (standard ThCr2Si2-type).
         No dedicated subfamily is created, since no literature vF value
         exists for this class either way -- both 'iron_1038' and
         'iron_other' are routed through the same ML fallback path, so a
         separate label adds no behavioral difference. Inserted BEFORE
         the iron_122 check.
      3. A15 element check extended with 'Sn' AND 'In' so V3Sn- and
         V3In-type stoichiometry (V + Sn or In, no Si/Ga needed) routes
         to 'conventional_A15' instead of falling through to
         'conventional_simple_metal'.
      4. FIX: cuprate_LSCO check now excludes compounds where Ba is
         present at MAJOR stoichiometry (>=0.5) TOGETHER WITH a rare
         earth from the YBCO RE list (Er, Gd, Sm, Nd, Ho, Dy, Eu, Lu, Pr,
         Tb, Tm, Yb) -- not just Y/Bi/Tl/Hg, and not RE-presence or
         trace-Ba presence alone. Previously, a genuine YBCO-type 123
         structure such as La0.5Er0.5Ba2Cu3O was caught by the La-only
         LSCO check before ever reaching the YBCO check below it, since
         the LSCO exclusion list did not include Er. The major-Ba
         threshold (matching the real ~2-per-formula-unit Ba2Cu3 block
         that defines 123-type structures) is required in addition to RE
         presence, because compounds such as La1.6-xNd0.4SrxCuO4
         ("Nd-LSCO") are a well-established, named branch of the 214-type
         LSCO family in the literature -- Nd doping on the La site, even
         alongside trace Ba doping, does not indicate a 123-type
         structure, and an RE-alone or any-Ba exclusion would have
         wrongly pulled these genuine LSCO compounds out of cuprate_LSCO.
      5. FIX: Bi+Sr (BSCCO) check now excludes compounds that also
         contain Hg. Previously, any compound containing both Bi and Hg
         (e.g. Hg0.5Bi0.5Sr2Ca2Cu3O8.35, a genuine mixed/hybrid
         composition) was unconditionally routed to cuprate_BSCCO, since
         the BSCCO check ran before the Hg-based check with no tie-break.
         Such hybrid compositions now route to cuprate_other rather than
         being silently assigned to a single parent structure.
      6. FIX: the nested Ce+In check inside the Cu+O cuprate block (used
         to catch Ce/In-based heavy-fermion superconductors that also
         happen to satisfy has('Cu') and has('O')) now carries the same
         not has('O') guard as the standalone heavy_fermion_115 check
         above it. Previously this nested check had no O guard, so any
         genuine Ce+In co-doped cuprate (e.g. an electron-doped compound
         such as Nd1.88Ce0.12Cu0.98In0.02O4) was misrouted to
         heavy_fermion_115 instead of cuprate_electron_doped.

    This function is unchanged in every other respect, including remaining
    fully case-sensitive as originally written. It must be called on
    ORIGINAL-CASE material strings (material_orig / orig_case_map), not on
    the lowercase 'material' column -- calling it on lowercase input will
    silently fail to match any element and return 'other' for everything.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    def has_explicit_coefficient_ge(element, string, threshold):
        # Stricter than has_major(): requires an EXPLICIT numeric
        # coefficient >= threshold. A bare element with no written number
        # (e.g. 'Se' in 'FeSe', implicit coefficient 1) does NOT
        # automatically pass here, unlike has_major(). Needed for
        # thresholds well above 1, where "no number written" must not be
        # treated as satisfying the threshold (see CHANGE 10 below).
        m = re.search(rf"{element}(?![a-z])([0-9]+\.?[0-9]*)", string)
        if m is None:
            return False
        return float(m.group(1)) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        # FIX (CHANGE 6): added not has('O', mat) guard, matching the
        # standalone heavy_fermion_115 check above. Without this guard,
        # any Ce+In co-doped cuprate (which always has O, by construction
        # of this block) was misrouted here instead of falling through to
        # the electron-doped cuprate check below.
        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        # FIX (CHANGE 4, updated): LSCO check now excludes compounds where
        # Ba is present at MAJOR stoichiometry (>=0.5, the real 123-type
        # Ba2Cu3-block signature) regardless of whether a second rare
        # earth is present. Literature (Ganguly & Sreedhar review of
        # RBa2Cu3O-delta phases) confirms pure LaBa2Cu3O-delta -- with La
        # as the ONLY rare earth, substituting partially onto the Ba site
        # -- is a genuine, documented member of the 123/YBCO family, not
        # a separate structure. An earlier version of this fix required a
        # SECOND rare earth (e.g. Er) alongside Ba, which correctly
        # excluded true 214-type "Nd-LSCO" compounds (La+Nd+Sr, only
        # trace Ba) but incorrectly still routed pure La+Ba2+Cu3+O7-type
        # compounds (no second RE) to cuprate_LSCO. The condition now
        # keys on major-Ba alone: trace-Ba Nd-LSCO compounds (Ba~0.14)
        # still correctly stay LSCO, since 0.14 < 0.5, while major-Ba
        # La-123 compounds (Ba~1.3-2) now correctly route to cuprate_YBCO.
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        # FIX (CHANGE 5): BSCCO check now excludes compounds that also
        # contain Hg, since Bi+Hg together indicates a genuine hybrid
        # composition rather than a clean BSCCO structure. Such compounds
        # now fall through to cuprate_other instead of being silently
        # assigned to BSCCO ahead of the Hg-based check below.
        if has('Bi', mat) and has('Sr', mat) and not has('Hg', mat):
            return 'cuprate_BSCCO'
        # FIX (CHANGE 8 continued): YBCO check's rare-earth trigger now
        # also includes La. Previously only Y and the RE list (Er, Gd,
        # Sm, Nd, Ho, Dy, Eu, Lu, Pr, Tb, Tm, Yb) reached this check --
        # La was excluded here too (it is handled as its own special
        # case throughout this function). Combined with the LSCO-check
        # fix above, a pure La+Ba(major)+Cu3+O7-type compound (e.g.
        # La1.5Ba1.3Ca0.2Cu3O6.897, no second rare earth) would otherwise
        # fall through BOTH the LSCO check (correctly excluded) and this
        # YBCO check (La not recognized as a trigger), landing in
        # cuprate_other instead of the correct cuprate_YBCO.
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        # NEW (CHANGE 2): Ca-Pt-arsenide "10-3-8"/"10-4-8" phase.
        # Requires Ca AND Pt together (the defining Ca10(Pt3As8)(Fe2As2)5
        # signature). Routed to iron_other (same ML-fallback path as
        # iron_1038 would have used) rather than a separate named
        # subfamily, since no literature vF value exists for this class.
        # Must come BEFORE the iron_1111 and iron_122 checks.
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 11): thick perovskite-blocking-layer families
        # (21113 e.g. Sr2VFeAsO3; 4226 e.g. Sr4V2Fe2As2O6, Ca4Al2Fe2As2O6;
        # 3225, 5229, 62212 and further homologues with n=3,4,5,6 perovskite
        # layers; 22241 e.g. Ba2Ti2Fe2As4O). Must come BEFORE the iron_1111
        # check, since all of these contain As+O and would otherwise be
        # caught there. Literature (Ogino et al., Appl. Phys. Lett. 97,
        # 072506, 2010; Nakamura & Machida, arXiv:1004.4741; Kotegawa et
        # al., arXiv:1009.5491; and the JACS report on Ba2Ti2Fe2As4O)
        # confirms these are a structurally distinct family from thin-layer
        # 1111 (LaFeAsO-type): the defining difference among all iron-based
        # numbering families is the composition and thickness of the
        # non-superconducting blocking layer between FeAs sheets, and this
        # family's blocking layer is a genuine multi-layer perovskite block
        # containing a SECOND transition/post-transition metal (V, Ti, Sc,
        # Al, Mg, or Cr) beyond Fe itself -- absent from true 1111, whose
        # blocking layer contains only the rare-earth/alkaline-earth cation
        # and O/H/F. No literature vF exists for this niche, structurally
        # complex family -- routed to iron_other (ML fallback).
        BLOCKING_LAYER_METALS = ['V', 'Ti', 'Sc', 'Al', 'Mg', 'Cr']
        if has('As', mat) and has('O', mat) and any(has(m, mat) for m in BLOCKING_LAYER_METALS):
            return 'iron_other'

        # NEW (CHANGE 12): 42210/42214 family (Ln4Fe2As2TeO4-type, e.g.
        # Pr4Fe2As2Te0.88O4, Sm4Fe2As2Te0.72F1.2O2.8). Must come BEFORE the
        # iron_1111 check for the same reason as CHANGE 11 (these contain
        # As+O). Literature (the 42214 structure report, arXiv:1311.5437)
        # confirms this family is isoconfigurational with the unrelated
        # La3Cu4P4O2/La3Ni4P4O2 ("3442") structure type -- built from
        # fluorite-type Ln2O2 layers and anti-fluorite-type Fe2As2 layers
        # stacked in a sequence and layer count genuinely different from
        # 1111's simple ZrCuSiAs stacking, and further distinguished by the
        # blocking layer's tellurium content. No literature vF exists --
        # routed to iron_other (ML fallback).
        if has('As', mat) and has('O', mat) and has('Te', mat):
            return 'iron_other'

        # FIX (CHANGE 8, extended): 'or has('F', mat)' added alongside H.
        # Literature (Matsuishi et al., J. Am. Chem. Soc. 130, 14428, 2008;
        # multiple independent AeFeAsF band-structure and synthesis papers)
        # confirms fluoroarsenide "1111" compounds (CaFeAsF, SrFeAsF, and
        # rare-earth-codoped variants such as Sr0.6La0.4FeAsF) adopt the
        # same ZrCuSiAs-type structure as the oxide 1111 family, with F-
        # substituting directly for O2- in the blocking layer -- the
        # isovalent counterpart to the H- substitution already handled
        # above. Without this, compounds like Ca1Fe0.9Co0.1As1F1 (no O or H)
        # failed this check and fell through to iron_122 instead, since Ca
        # then triggered that branch's alkaline-earth condition. The two
        # thick-blocking-layer checks above (CHANGE 11, CHANGE 12) run
        # first so that genuine thin-layer 1111 compounds (no second
        # transition metal, no Te) are the only ones reaching this check.
        if has('As', mat) and (has('O', mat) or has('H', mat) or has('F', mat)):
            return 'iron_1111'

        # NEW (CHANGE 10): quasi-1D "ladder" AFe2Se3 phase (A = Ba, K, Cs).
        # Must come BEFORE both the iron_245 and iron_11 checks below, since
        # these compounds would otherwise be caught by either (they contain
        # the same alkali/alkaline-earth intercalants as iron_245, and no
        # As, same as iron_11). Literature (Krzton-Maziopa et al., J. Phys.
        # Condens. Matter 23, 402201, 2011; Caron et al., PRB 84, 180409,
        # 2011; Nature Materials 14, 745, 2015) confirms AFe2Se3 is a
        # structurally DISTINCT family from both: built from quasi-1D
        # two-leg Fe-Se ladders (edge-sharing FeSe4 tetrahedra), not the 2D
        # FeSe layers found in either iron_11 (simple binary FeSe/FeTe) or
        # iron_245 (2D AFe2Se2 intercalated layers). Detected here by
        # Fe:Se stoichiometry near 2:3 (vs 1:1 for iron_11, 1:2 for
        # iron_245). No literature vF exists for this niche, primarily
        # magnetically-studied family -- routed to iron_other (ML fallback)
        # rather than a hand-picked or borrowed lookup value.
        if has_explicit_coefficient_ge('Se', mat, 2.5) and not has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 1, extended): alkali- OR alkaline-earth-intercalated
        # 245-type selenide. Ca/Sr/Ba/Eu/Yb added alongside the original
        # K/Rb/Cs/Tl/Na alkali list -- literature confirms AFe2Se2 (A = Ca,
        # Sr, Ba, Eu, Yb, in addition to the alkalis) shares the same
        # ThCr2Si2-type "122-like" intercalated structure, distinct from
        # simple binary FeSe/FeTe (iron_11). Detected by Fe:Se ~= 1:2
        # stoichiometry (the ladder-compound check above, which requires
        # Se >= 2.5, already excludes the 2:3-stoichiometry AFe2Se3 ladder
        # compounds from reaching this branch).
        if any(has(a, mat) for a in ['Li', 'K', 'Rb', 'Cs', 'Tl', 'Na', 'Ca', 'Sr', 'Ba', 'Eu', 'Yb']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    # MODIFIED (CHANGE 3): added 'Sn', 'In', and 'Al'. V-Al alloys are
    # explicitly documented A15-structure superconductors (see e.g.
    # Zeitschrift fur Physik B, "Critical temperature ... of superconducting
    # V-Ga-Al alloys with A15 structure").
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    # Was 'conventional_NbTi'. Renamed to 'conventional_Nb_based' since the
    # actual contents of this bucket are general Nb binary/ternary
    # compounds (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2),
    # not specifically Nb-Ti alloys -- the old name implied a narrower,
    # inaccurate structural claim.
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    # BaBiO3-derived perovskite/CDW superconductors: BaPb1-xBixO3 (BPBO) and
    # Ba1-xKxBiO3 (BKBO). Structurally and mechanistically distinct from the
    # sp-band alloys in conventional_simple_metal (non-magnetic CDW-derived
    # perovskite oxide, not a simple metallic alloy) -- see e.g. Nature
    # Materials/Nature Communications reviews of the BPBO/BKBO family.
    # Must come BEFORE the simple-metals check, since Pb/Al/In there would
    # otherwise catch these compounds first.
    # No literature vF value is assigned for this class -- routed to
    # 'other' (ML fallback), matching how iron_1038 and other
    # structurally-distinct-but-unvalued classes are handled.
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# vF LOOKUP TABLE -- direct-lookup subfamilies only.
#
# IMPORTANT: 'other', 'iron_other', and 'cuprate_other' are INTENTIONALLY
# ABSENT from this dictionary. These are the catch-all buckets for
# structurally ambiguous or genuinely mixed compounds (e.g. the Ca-Pt-As
# "10-3-8" phase inside iron_other, Hg+Bi hybrid cuprates inside
# cuprate_other, and chemically unrelated compounds like bismuthate
# perovskites or Fe-Ni-Zr Laves phases inside the generic 'other'). No
# single literature-backed Fermi velocity exists for these buckets, since
# they are deliberately heterogeneous by construction, not a real physical
# family. Per the actual training pipeline (vF_xi_Hc2.ipynb, Section 2:
# AMBIGUOUS_SUBFAMILIES / is_ambiguous_row), rows in these three subfamilies
# are ALWAYS routed to the trained ML fallback model (final_model.predict,
# followed by snap_to_valid) instead of a fixed lookup value. An earlier
# version of this file incorrectly gave these three subfamilies a borrowed
# or arbitrary constant (cuprate_other = cuprate_YBCO's value, iron_other =
# iron_122's value, other = an unexplained 760,000 default) -- these were
# never real citations and did not match how the actual pipeline behaves,
# so get_vF() below now raises rather than silently returning a number that
# looks citation-backed but isn't.
#
# conventional_A15 updated to the literature value: Mentink et al.,
# Supercond. Sci. Technol. 30, 025006 (2017) -- ab initio DFT calculation
# of Nb3Sn, RMS Fermi velocity vF = 2.121e5 m/s, cross-validated against
# four independent experimental sources (Lim, Schachinger, Orlando, Ghosh)
# in the same paper. Replaces the previous unsourced 850,000 m/s.
#
# conventional_Nb_based updated to: Prozorov, Zarea, and Sauls, arXiv:
# 2207.11829 (2022), citing the underlying DFT calculation of elemental Nb
# by Zarea, Ueki, and Sauls, arXiv:2201.07403 (2022). Two-band-averaged
# RMS Fermi velocity v = 7.26e5 m/s. This is elemental niobium's own value,
# used as the representative constant for the whole conventional_Nb_based
# bucket (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2, etc.)
# since Nb is the one element common to every compound in this bucket;
# compound-specific corrections (carbide vs boride vs oxide bonding) have
# no citable literature basis and are not applied.
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'iron_1111':                 152000,
    'iron_122':                  91981,
    'iron_111':                  45600,
    'iron_11':                    13700,
    'iron_245':                   79000,   # Qian et al., PRL 106, 187001 (2011)
    'MgB2_type':                 560000,
    'conventional_A15':          212100,   # Mentink et al., SST 30, 025006 (2017)
    'conventional_Nb_based':     726000,   # Prozorov, Zarea & Sauls, arXiv:2207.11829 (2022)
    'conventional_simple_metal': 1800000,
    'borocarbide':               276000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    # 'cuprate_other', 'iron_other', 'other' deliberately absent -- see above.
}

# Subfamilies with no literature-backed vF, routed to the trained ML
# fallback model instead of VF_LOOKUP. Matches AMBIGUOUS_SUBFAMILIES in
# vF_xi_Hc2.ipynb exactly.
# AMBIGUOUS_SUBFAMILIES = {"other", "iron_other", "cuprate_other"}

# def is_ambiguous_row(subfamily, low_confidence=False):
#     """True if this row must go through the ML fallback instead of direct lookup."""
#     return (subfamily in AMBIGUOUS_SUBFAMILIES) or bool(low_confidence)

# def get_vF(mat_orig, low_confidence=False):
#     """
#     Direct-lookup Fermi velocity for non-ambiguous subfamilies only.

#     Raises ValueError for 'other', 'iron_other', 'cuprate_other', or any
#     row flagged low_confidence, since these have no literature-backed
#     lookup value by design -- they must be scored by the trained ML
#     fallback model instead (see vF_xi_Hc2.ipynb, Section 2 and Section 10).
#     This function intentionally does NOT silently return a placeholder for
#     these cases, since an earlier version's silent fallback (a borrowed or
#     arbitrary constant) is not something you can defend to a reviewer as a
#     physical value.
#     """
#     sf = get_subfamily(mat_orig)
#     if is_ambiguous_row(sf, low_confidence):
#         raise ValueError(
#             f"'{sf}' has no literature-backed vF lookup value -- route this "
#             f"compound through the trained ML fallback model instead."
#         )
#     return VF_LOOKUP[sf]
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

In [2]:
## New version v4
import re

def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.

    CHANGE LOG (relative to fermi_velocity_169_fixed_1.ipynb):
      1. NEW: alkali-intercalated (K/Rb/Cs/Tl) FeSe-derived compounds
         (e.g. K0.8Fe1.7Se2) now route to 'iron_245' instead of being
         silently absorbed into 'iron_11' (simple binary FeSe/FeTe).
         Inserted BEFORE the iron_11 check.
      2. Ca-Pt-intercalated Fe-As "10-3-8"/"10-4-8" phase compounds
         (e.g. Ca10Pt3As8(Fe2As2)5) now route to 'iron_other' instead of
         being silently absorbed into 'iron_122' (standard ThCr2Si2-type).
         No dedicated subfamily is created, since no literature vF value
         exists for this class either way -- both 'iron_1038' and
         'iron_other' are routed through the same ML fallback path, so a
         separate label adds no behavioral difference. Inserted BEFORE
         the iron_122 check.
      3. A15 element check extended with 'Sn' AND 'In' so V3Sn- and
         V3In-type stoichiometry (V + Sn or In, no Si/Ga needed) routes
         to 'conventional_A15' instead of falling through to
         'conventional_simple_metal'.
      4. FIX: cuprate_LSCO check now excludes compounds where Ba is
         present at MAJOR stoichiometry (>=0.5) TOGETHER WITH a rare
         earth from the YBCO RE list (Er, Gd, Sm, Nd, Ho, Dy, Eu, Lu, Pr,
         Tb, Tm, Yb) -- not just Y/Bi/Tl/Hg, and not RE-presence or
         trace-Ba presence alone. Previously, a genuine YBCO-type 123
         structure such as La0.5Er0.5Ba2Cu3O was caught by the La-only
         LSCO check before ever reaching the YBCO check below it, since
         the LSCO exclusion list did not include Er. The major-Ba
         threshold (matching the real ~2-per-formula-unit Ba2Cu3 block
         that defines 123-type structures) is required in addition to RE
         presence, because compounds such as La1.6-xNd0.4SrxCuO4
         ("Nd-LSCO") are a well-established, named branch of the 214-type
         LSCO family in the literature -- Nd doping on the La site, even
         alongside trace Ba doping, does not indicate a 123-type
         structure, and an RE-alone or any-Ba exclusion would have
         wrongly pulled these genuine LSCO compounds out of cuprate_LSCO.
      5. FIX: Bi+Sr (BSCCO) check now excludes compounds that also
         contain Hg. Previously, any compound containing both Bi and Hg
         (e.g. Hg0.5Bi0.5Sr2Ca2Cu3O8.35, a genuine mixed/hybrid
         composition) was unconditionally routed to cuprate_BSCCO, since
         the BSCCO check ran before the Hg-based check with no tie-break.
         Such hybrid compositions now route to cuprate_other rather than
         being silently assigned to a single parent structure.
      6. FIX: the nested Ce+In check inside the Cu+O cuprate block (used
         to catch Ce/In-based heavy-fermion superconductors that also
         happen to satisfy has('Cu') and has('O')) now carries the same
         not has('O') guard as the standalone heavy_fermion_115 check
         above it. Previously this nested check had no O guard, so any
         genuine Ce+In co-doped cuprate (e.g. an electron-doped compound
         such as Nd1.88Ce0.12Cu0.98In0.02O4) was misrouted to
         heavy_fermion_115 instead of cuprate_electron_doped.

    This function is unchanged in every other respect, including remaining
    fully case-sensitive as originally written. It must be called on
    ORIGINAL-CASE material strings (material_orig / orig_case_map), not on
    the lowercase 'material' column -- calling it on lowercase input will
    silently fail to match any element and return 'other' for everything.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    def has_explicit_coefficient_ge(element, string, threshold):
        # Stricter than has_major(): requires an EXPLICIT numeric
        # coefficient >= threshold. A bare element with no written number
        # (e.g. 'Se' in 'FeSe', implicit coefficient 1) does NOT
        # automatically pass here, unlike has_major(). Needed for
        # thresholds well above 1, where "no number written" must not be
        # treated as satisfying the threshold (see CHANGE 10 below).
        m = re.search(rf"{element}(?![a-z])([0-9]+\.?[0-9]*)", string)
        if m is None:
            return False
        return float(m.group(1)) >= threshold

    def get_coeff(element, string):
        # Returns the stoichiometric coefficient of an element (1.0 if
        # present with no explicit number, 0.0 if absent). Used for
        # direct coefficient-vs-coefficient comparisons between two
        # elements (see CHANGE 13), where a simple has()/has_major()
        # presence check is not enough to determine which element is
        # structurally dominant.
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return 0.0
        num = m.group(1)
        return 1.0 if num == '' else float(num)

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        # FIX (CHANGE 6): added not has('O', mat) guard, matching the
        # standalone heavy_fermion_115 check above. Without this guard,
        # any Ce+In co-doped cuprate (which always has O, by construction
        # of this block) was misrouted here instead of falling through to
        # the electron-doped cuprate check below.
        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        # FIX (CHANGE 4, updated): LSCO check now excludes compounds where
        # Ba is present at MAJOR stoichiometry (>=0.5, the real 123-type
        # Ba2Cu3-block signature) regardless of whether a second rare
        # earth is present. Literature (Ganguly & Sreedhar review of
        # RBa2Cu3O-delta phases) confirms pure LaBa2Cu3O-delta -- with La
        # as the ONLY rare earth, substituting partially onto the Ba site
        # -- is a genuine, documented member of the 123/YBCO family, not
        # a separate structure. An earlier version of this fix required a
        # SECOND rare earth (e.g. Er) alongside Ba, which correctly
        # excluded true 214-type "Nd-LSCO" compounds (La+Nd+Sr, only
        # trace Ba) but incorrectly still routed pure La+Ba2+Cu3+O7-type
        # compounds (no second RE) to cuprate_LSCO. The condition now
        # keys on major-Ba alone: trace-Ba Nd-LSCO compounds (Ba~0.14)
        # still correctly stay LSCO, since 0.14 < 0.5, while major-Ba
        # La-123 compounds (Ba~1.3-2) now correctly route to cuprate_YBCO.
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        # FIX (CHANGE 5, revised) + FIX (CHANGE 13): BSCCO check now
        # requires Bi's coefficient to strictly exceed both Hg's and Tl's
        # coefficients, rather than excluding on bare presence of either
        # element. Literature confirms both Bi2-xHgxSr2-yBayCaCu2O8-type
        # compounds (Toulemonde and Odier, arXiv:cond-mat/0304373; Jabur
        # et al., Eng. Technol. J. 31, 2650, 2013) and Tl1-xBix-1223/1212/
        # 1201-type compounds (Hur et al., Physica C 253, 109, 1995; Lee
        # et al., J. Appl. Phys. 80, 326, 1996) are treated in the
        # primary literature as substitutional variants of their
        # respective PARENT structures (Bi-2212 for the Hg case, Tl-1223/
        # 1212/1201 for the Tl case), with the parent's defining element
        # remaining structurally dominant at substitution levels up to
        # ~50%. STRICT inequality (not >=) is used deliberately: at an
        # exact 1:1 Hg:Bi ratio the compound is treated as Hg-based (its
        # own established convention elsewhere in this function), while
        # at an exact 1:1 Tl:Bi ratio the compound is treated as Tl-based
        # (e.g. "Tl-1201", "Tl-1212" oxycarbonate phases named as such in
        # the primary literature regardless of the 1:1 substitution
        # level) -- so Bi must strictly exceed each competing element's
        # coefficient, not merely equal it, for BSCCO to apply.
        if has('Bi', mat) and has('Sr', mat) and get_coeff('Bi', mat) > get_coeff('Hg', mat) and get_coeff('Bi', mat) > get_coeff('Tl', mat):
            return 'cuprate_BSCCO'
        # FIX (CHANGE 8 continued): YBCO check's rare-earth trigger now
        # also includes La. Previously only Y and the RE list (Er, Gd,
        # Sm, Nd, Ho, Dy, Eu, Lu, Pr, Tb, Tm, Yb) reached this check --
        # La was excluded here too (it is handled as its own special
        # case throughout this function). Combined with the LSCO-check
        # fix above, a pure La+Ba(major)+Cu3+O7-type compound (e.g.
        # La1.5Ba1.3Ca0.2Cu3O6.897, no second rare earth) would otherwise
        # fall through BOTH the LSCO check (correctly excluded) and this
        # YBCO check (La not recognized as a trigger), landing in
        # cuprate_other instead of the correct cuprate_YBCO.
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        # NEW (CHANGE 2): Ca-Pt-arsenide "10-3-8"/"10-4-8" phase.
        # Requires Ca AND Pt together (the defining Ca10(Pt3As8)(Fe2As2)5
        # signature). Routed to iron_other (same ML-fallback path as
        # iron_1038 would have used) rather than a separate named
        # subfamily, since no literature vF value exists for this class.
        # Must come BEFORE the iron_1111 and iron_122 checks.
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 11): thick perovskite-blocking-layer families
        # (21113 e.g. Sr2VFeAsO3; 4226 e.g. Sr4V2Fe2As2O6, Ca4Al2Fe2As2O6;
        # 3225, 5229, 62212 and further homologues with n=3,4,5,6 perovskite
        # layers; 22241 e.g. Ba2Ti2Fe2As4O). Must come BEFORE the iron_1111
        # check, since all of these contain As+O and would otherwise be
        # caught there. Literature (Ogino et al., Appl. Phys. Lett. 97,
        # 072506, 2010; Nakamura & Machida, arXiv:1004.4741; Kotegawa et
        # al., arXiv:1009.5491; and the JACS report on Ba2Ti2Fe2As4O)
        # confirms these are a structurally distinct family from thin-layer
        # 1111 (LaFeAsO-type): the defining difference among all iron-based
        # numbering families is the composition and thickness of the
        # non-superconducting blocking layer between FeAs sheets, and this
        # family's blocking layer is a genuine multi-layer perovskite block
        # containing a SECOND transition/post-transition metal (V, Ti, Sc,
        # Al, Mg, or Cr) beyond Fe itself -- absent from true 1111, whose
        # blocking layer contains only the rare-earth/alkaline-earth cation
        # and O/H/F. No literature vF exists for this niche, structurally
        # complex family -- routed to iron_other (ML fallback).
        BLOCKING_LAYER_METALS = ['V', 'Ti', 'Sc', 'Al', 'Mg', 'Cr']
        if has('As', mat) and has('O', mat) and any(has(m, mat) for m in BLOCKING_LAYER_METALS):
            return 'iron_other'

        # NEW (CHANGE 12): 42210/42214 family (Ln4Fe2As2TeO4-type, e.g.
        # Pr4Fe2As2Te0.88O4, Sm4Fe2As2Te0.72F1.2O2.8). Must come BEFORE the
        # iron_1111 check for the same reason as CHANGE 11 (these contain
        # As+O). Literature (the 42214 structure report, arXiv:1311.5437)
        # confirms this family is isoconfigurational with the unrelated
        # La3Cu4P4O2/La3Ni4P4O2 ("3442") structure type -- built from
        # fluorite-type Ln2O2 layers and anti-fluorite-type Fe2As2 layers
        # stacked in a sequence and layer count genuinely different from
        # 1111's simple ZrCuSiAs stacking, and further distinguished by the
        # blocking layer's tellurium content. No literature vF exists --
        # routed to iron_other (ML fallback).
        if has('As', mat) and has('O', mat) and has('Te', mat):
            return 'iron_other'

        # FIX (CHANGE 8, extended): 'or has('F', mat)' added alongside H.
        # Literature (Matsuishi et al., J. Am. Chem. Soc. 130, 14428, 2008;
        # multiple independent AeFeAsF band-structure and synthesis papers)
        # confirms fluoroarsenide "1111" compounds (CaFeAsF, SrFeAsF, and
        # rare-earth-codoped variants such as Sr0.6La0.4FeAsF) adopt the
        # same ZrCuSiAs-type structure as the oxide 1111 family, with F-
        # substituting directly for O2- in the blocking layer -- the
        # isovalent counterpart to the H- substitution already handled
        # above. Without this, compounds like Ca1Fe0.9Co0.1As1F1 (no O or H)
        # failed this check and fell through to iron_122 instead, since Ca
        # then triggered that branch's alkaline-earth condition. The two
        # thick-blocking-layer checks above (CHANGE 11, CHANGE 12) run
        # first so that genuine thin-layer 1111 compounds (no second
        # transition metal, no Te) are the only ones reaching this check.
        if has('As', mat) and (has('O', mat) or has('H', mat) or has('F', mat)):
            return 'iron_1111'

        # NEW (CHANGE 10): quasi-1D "ladder" AFe2Se3 phase (A = Ba, K, Cs).
        # Must come BEFORE both the iron_245 and iron_11 checks below, since
        # these compounds would otherwise be caught by either (they contain
        # the same alkali/alkaline-earth intercalants as iron_245, and no
        # As, same as iron_11). Literature (Krzton-Maziopa et al., J. Phys.
        # Condens. Matter 23, 402201, 2011; Caron et al., PRB 84, 180409,
        # 2011; Nature Materials 14, 745, 2015) confirms AFe2Se3 is a
        # structurally DISTINCT family from both: built from quasi-1D
        # two-leg Fe-Se ladders (edge-sharing FeSe4 tetrahedra), not the 2D
        # FeSe layers found in either iron_11 (simple binary FeSe/FeTe) or
        # iron_245 (2D AFe2Se2 intercalated layers). Detected here by
        # Fe:Se stoichiometry near 2:3 (vs 1:1 for iron_11, 1:2 for
        # iron_245). No literature vF exists for this niche, primarily
        # magnetically-studied family -- routed to iron_other (ML fallback)
        # rather than a hand-picked or borrowed lookup value.
        if has_explicit_coefficient_ge('Se', mat, 2.5) and not has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 1, extended): alkali- OR alkaline-earth-intercalated
        # 245-type selenide. Ca/Sr/Ba/Eu/Yb added alongside the original
        # K/Rb/Cs/Tl/Na alkali list -- literature confirms AFe2Se2 (A = Ca,
        # Sr, Ba, Eu, Yb, in addition to the alkalis) shares the same
        # ThCr2Si2-type "122-like" intercalated structure, distinct from
        # simple binary FeSe/FeTe (iron_11). Detected by Fe:Se ~= 1:2
        # stoichiometry (the ladder-compound check above, which requires
        # Se >= 2.5, already excludes the 2:3-stoichiometry AFe2Se3 ladder
        # compounds from reaching this branch).
        if any(has(a, mat) for a in ['Li', 'K', 'Rb', 'Cs', 'Tl', 'Na', 'Ca', 'Sr', 'Ba', 'Eu', 'Yb']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    # MODIFIED (CHANGE 3): added 'Sn', 'In', and 'Al'. V-Al alloys are
    # explicitly documented A15-structure superconductors (see e.g.
    # Zeitschrift fur Physik B, "Critical temperature ... of superconducting
    # V-Ga-Al alloys with A15 structure").
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    # Was 'conventional_NbTi'. Renamed to 'conventional_Nb_based' since the
    # actual contents of this bucket are general Nb binary/ternary
    # compounds (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2),
    # not specifically Nb-Ti alloys -- the old name implied a narrower,
    # inaccurate structural claim.
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    # BaBiO3-derived perovskite/CDW superconductors: BaPb1-xBixO3 (BPBO) and
    # Ba1-xKxBiO3 (BKBO). Structurally and mechanistically distinct from the
    # sp-band alloys in conventional_simple_metal (non-magnetic CDW-derived
    # perovskite oxide, not a simple metallic alloy) -- see e.g. Nature
    # Materials/Nature Communications reviews of the BPBO/BKBO family.
    # Must come BEFORE the simple-metals check, since Pb/Al/In there would
    # otherwise catch these compounds first.
    # No literature vF value is assigned for this class -- routed to
    # 'other' (ML fallback), matching how iron_1038 and other
    # structurally-distinct-but-unvalued classes are handled.
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# vF LOOKUP TABLE -- direct-lookup subfamilies only.
#
# IMPORTANT: 'other', 'iron_other', and 'cuprate_other' are INTENTIONALLY
# ABSENT from this dictionary. These are the catch-all buckets for
# structurally ambiguous or genuinely mixed compounds (e.g. the Ca-Pt-As
# "10-3-8" phase inside iron_other, Hg+Bi hybrid cuprates inside
# cuprate_other, and chemically unrelated compounds like bismuthate
# perovskites or Fe-Ni-Zr Laves phases inside the generic 'other'). No
# single literature-backed Fermi velocity exists for these buckets, since
# they are deliberately heterogeneous by construction, not a real physical
# family. Per the actual training pipeline (vF_xi_Hc2.ipynb, Section 2:
# AMBIGUOUS_SUBFAMILIES / is_ambiguous_row), rows in these three subfamilies
# are ALWAYS routed to the trained ML fallback model (final_model.predict,
# followed by snap_to_valid) instead of a fixed lookup value. An earlier
# version of this file incorrectly gave these three subfamilies a borrowed
# or arbitrary constant (cuprate_other = cuprate_YBCO's value, iron_other =
# iron_122's value, other = an unexplained 760,000 default) -- these were
# never real citations and did not match how the actual pipeline behaves,
# so get_vF() below now raises rather than silently returning a number that
# looks citation-backed but isn't.
#
# conventional_A15 updated to the literature value: Mentink et al.,
# Supercond. Sci. Technol. 30, 025006 (2017) -- ab initio DFT calculation
# of Nb3Sn, RMS Fermi velocity vF = 2.121e5 m/s, cross-validated against
# four independent experimental sources (Lim, Schachinger, Orlando, Ghosh)
# in the same paper. Replaces the previous unsourced 850,000 m/s.
#
# conventional_Nb_based updated to: Prozorov, Zarea, and Sauls, arXiv:
# 2207.11829 (2022), citing the underlying DFT calculation of elemental Nb
# by Zarea, Ueki, and Sauls, arXiv:2201.07403 (2022). Two-band-averaged
# RMS Fermi velocity v = 7.26e5 m/s. This is elemental niobium's own value,
# used as the representative constant for the whole conventional_Nb_based
# bucket (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2, etc.)
# since Nb is the one element common to every compound in this bucket;
# compound-specific corrections (carbide vs boride vs oxide bonding) have
# no citable literature basis and are not applied.
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'iron_1111':                 152000,
    'iron_122':                  91981,
    'iron_111':                  45600,
    'iron_11':                    13700,
    'iron_245':                   79000,   # Qian et al., PRL 106, 187001 (2011)
    'MgB2_type':                 560000,
    'conventional_A15':          212100,   # Mentink et al., SST 30, 025006 (2017)
    'conventional_Nb_based':     726000,   # Prozorov, Zarea & Sauls, arXiv:2207.11829 (2022)
    'conventional_simple_metal': 1800000,
    'borocarbide':               276000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    # 'cuprate_other', 'iron_other', 'other' deliberately absent -- see above.
}

# Subfamilies with no literature-backed vF, routed to the trained ML
# fallback model instead of VF_LOOKUP. Matches AMBIGUOUS_SUBFAMILIES in
# vF_xi_Hc2.ipynb exactly.
# AMBIGUOUS_SUBFAMILIES = {"other", "iron_other", "cuprate_other"}

# def is_ambiguous_row(subfamily, low_confidence=False):
#     """True if this row must go through the ML fallback instead of direct lookup."""
#     return (subfamily in AMBIGUOUS_SUBFAMILIES) or bool(low_confidence)

# def get_vF(mat_orig, low_confidence=False):
#     """
#     Direct-lookup Fermi velocity for non-ambiguous subfamilies only.

#     Raises ValueError for 'other', 'iron_other', 'cuprate_other', or any
#     row flagged low_confidence, since these have no literature-backed
#     lookup value by design -- they must be scored by the trained ML
#     fallback model instead (see vF_xi_Hc2.ipynb, Section 2 and Section 10).
#     This function intentionally does NOT silently return a placeholder for
#     these cases, since an earlier version's silent fallback (a borrowed or
#     arbitrary constant) is not something you can defend to a reviewer as a
#     physical value.
#     """
#     sf = get_subfamily(mat_orig)
#     if is_ambiguous_row(sf, low_confidence):
#         raise ValueError(
#             f"'{sf}' has no literature-backed vF lookup value -- route this "
#             f"compound through the trained ML fallback model instead."
#         )
#     return VF_LOOKUP[sf]

def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

In [3]:
 # Sanity check — runs when cell executes
_tests = [
    ('Mg0.95Fe0.05B2',          'MgB2_type',              560000),
    ('MgB2',                    'MgB2_type',              560000),
    ('CeIn3',                   'heavy_fermion_115',        7595),
    ('CeCoIn5',                 'heavy_fermion_115',        7595),
    ('CeCu2Si2',                'heavy_fermion_Ce',         7595),
    ('Tl1.4Hg0.6Ba2Ca2Cu3O',   'cuprate_Tl_based',       320000),
    ('HgBa2CuO4',               'cuprate_Hg_based',       290000),
    ('Ca0.5Na0.5Fe2As2',        'iron_122',               91981),
    ('Sn0.97As0.022Te0.978',    'chalcogenide',            95000),
    ('PbMo6S8',                 'chalcogenide',            95000),
    ('Nd1.47La0.4Ce0.13Cu1O4',  'cuprate_electron_doped', 230000),
    ('Nd1.85Ce0.15CuO4',        'cuprate_electron_doped', 230000),
    ('YBa2Cu3O7',               'cuprate_YBCO',           273400),
    ('La1.85Sr0.15CuO4',        'cuprate_LSCO',           245000),
    ('Bi2Sr2CaCu2O8',           'cuprate_BSCCO',          304000),
    ('Nb3Sn',                   'conventional_A15',       212100),
    ('FeSe',                    'iron_11',                 13700),
    ('LiFeAs',                  'iron_111',               45600),
    ('LuNi2B2C',                'borocarbide',            276000),
]
 
print("=== STEP 2 SANITY CHECK ===\n")
_bugs = 0
for _mat, _exp_sf, _exp_vf in _tests:
    _got_sf = get_subfamily(_mat)
    _got_vf = get_vF(_mat)
    _sf_ok  = 'OK ' if _got_sf == _exp_sf else 'BUG'
    _vf_ok  = 'OK ' if _got_vf == _exp_vf else 'BUG'
    if 'BUG' in (_sf_ok, _vf_ok):
        _bugs += 1
    print(f"  {_sf_ok} {_vf_ok} | {_mat:40} | {_got_sf:30} | vF: {_got_vf:>8,}")
 
print(f"\n  >>> {'ALL CLEAR — 0 bugs' if _bugs == 0 else f'{_bugs} BUGS REMAINING'}\n")
 


=== STEP 2 SANITY CHECK ===

  OK  OK  | Mg0.95Fe0.05B2                           | MgB2_type                      | vF:  560,000
  OK  OK  | MgB2                                     | MgB2_type                      | vF:  560,000
  OK  OK  | CeIn3                                    | heavy_fermion_115              | vF:    7,595
  OK  OK  | CeCoIn5                                  | heavy_fermion_115              | vF:    7,595
  OK  OK  | CeCu2Si2                                 | heavy_fermion_Ce               | vF:    7,595
  OK  OK  | Tl1.4Hg0.6Ba2Ca2Cu3O                     | cuprate_Tl_based               | vF:  320,000
  OK  OK  | HgBa2CuO4                                | cuprate_Hg_based               | vF:  290,000
  OK  OK  | Ca0.5Na0.5Fe2As2                         | iron_122                       | vF:   91,981
  OK  OK  | Sn0.97As0.022Te0.978                     | chalcogenide                   | vF:   95,000
  OK  OK  | PbMo6S8                                  | chalcog

In [4]:

# =============================================================================
# STEP 3 — Build df_combine: all 21,263 superconductors
# =============================================================================
df_combine = df_train.merge(
    df_unique_m[['material_lower', 'material_orig']],
    left_index=True, right_index=True,
    how='inner'
)
df_combine['subfamily'] = df_combine['material_orig'].apply(get_subfamily)
df_combine['family']    = df_combine['material_orig'].apply(get_family)
df_combine['low_confidence'] = df_combine['material_orig'].apply(is_low_confidence_extrapolation)  # FIX-9
# df_combine['low_confidence'] = df_combine['material_orig'].apply(is_ambiguous_row)  # FIX-9
df_combine['material']  = df_combine['material_lower']
df_combine.drop(columns=['material_lower', 'material_orig'], inplace=True)

print(f"df_combine shape : {df_combine.shape}")
df_combine.to_csv("UCI_family_subFamily.csv", index=False)
print("Saved: UCI_family_subFamily.csv")

print("\nFamily distribution (df_combine):")
print(df_combine['family'].value_counts().to_string())

print("\nSubfamily distribution (df_combine):")
print(df_combine['subfamily'].value_counts().to_string())

print(f"\nLow-confidence (extrapolated) rows: {df_combine['low_confidence'].sum()} ({df_combine['low_confidence'].mean():.2%})")


df_combine shape : (21263, 86)
Saved: UCI_family_subFamily.csv

Family distribution (df_combine):
family
Cuprates                 10531
Conventional / Alloys     7121
Iron-based                1801
Chalcogenide              1161
Mg-B related               336
Heavy Fermion              313

Subfamily distribution (df_combine):
subfamily
cuprate_YBCO                 4811
other                        4277
cuprate_BSCCO                1657
cuprate_LSCO                 1300
chalcogenide                 1161
cuprate_other                1064
conventional_Nb_based         923
conventional_simple_metal     852
conventional_A15              728
cuprate_Tl_based              626
iron_122                      611
cuprate_Hg_based              609
iron_1111                     568
cuprate_electron_doped        464
borocarbide                   341
MgB2_type                     336
iron_other                    243
iron_11                       195
heavy_fermion_Ce              166
heavy_fermion_1

In [5]:
# =============================================================================
# STEP 4 — Build df_169: the 169-sample training set with family + subfamily features
# =============================================================================
df_759 = df_fermi.copy()

# Re-derive subfamily and family from original-case material names
# (overwrites the stored subfamily to ensure consistency with df_combine)
# df_759['material_orig'] = df_759['material'].map(orig_case_map)
# df_759['subfamily']     = df_759['material_orig'].apply(get_subfamily)
# df_759['family']        = df_759['material_orig'].apply(get_family)

# Add this at the end of STEP 4, after df_759 is built
df_759['material_orig'] = df_759['material'].map(orig_case_map)
df_759['vF (m/s)'] = df_759['material_orig'].apply(get_vF)   # ← reassign from VF_LOOKUP
df_759['subfamily'] = df_759['material_orig'].apply(get_subfamily)
df_759['family']    = df_759['material_orig'].apply(get_family)
df_759.drop(columns=['material_orig'], inplace=True)

print(f"\ndf_759 shape     : {df_759.shape}")
print(f"Unique materials : {df_759['material'].nunique()}")
print(f"Unique vF values : {df_759['vF (m/s)'].nunique()}")
print("\nSubfamily distribution (df_759):")
print(df_759['subfamily'].value_counts().to_string())
print(df_759.columns)
# df_759.to_csv("Fermi_759_all_features.csv", index=False)
df_759.to_csv("Fermi_169_unique_v6.csv", index=False)


df_759 shape     : (162, 5)
Unique materials : 162
Unique vF values : 19

Subfamily distribution (df_759):
subfamily
cuprate_YBCO                 26
cuprate_LSCO                 11
cuprate_BSCCO                11
iron_1111                    11
cuprate_Hg_based             10
cuprate_Tl_based             10
iron_122                     10
cuprate_electron_doped        9
conventional_Nb_based         8
MgB2_type                     8
conventional_A15              8
conventional_simple_metal     7
iron_111                      6
chalcogenide                  6
iron_11                       5
borocarbide                   5
heavy_fermion_Ce              3
heavy_fermion_115             3
iron_245                      3
heavy_fermion_Yb              2
Index(['material', 'critical_temp', 'vF (m/s)', 'subfamily', 'family'], dtype='object')


In [6]:
# import pandas as pd
# df_uci_family_subfamily = pd.read_csv("UCI_family_subFamily.csv")
# # df_fermi = pd.read_csv("Fermi_169_unique_v6.csv")
# df_vf = pd.merge(df_759, df_uci_family_subfamily, on='material', how='inner')
# print(f"\ndf_vf shape     : {df_vf.shape}")
# print(df_vf.head(10))
import pandas as pd

# 1. Load datasets
# (Update file names if your file is 'Fermi_169_unique_v6_3.csv')
fermi_file = "Fermi_169_unique_v6.csv"
uci_file = "UCI_family_subFamily.csv"

fermi_df = pd.read_csv(fermi_file)
uci_df = pd.read_csv(uci_file)

# 2. Specify the column name used to identify compounds in both files
# Replace 'material' with your actual column name (e.g., 'Compound', 'material_id')
compound_col = "material"

# Clean up any leading/trailing whitespace in compound names
fermi_df[compound_col] = fermi_df[compound_col].astype(str).str.strip()
uci_df[compound_col] = uci_df[compound_col].astype(str).str.strip()

# 3. Merge features with your fixed vf values
# This keeps all columns from Fermi dataset and attaches all 81 features from UCI dataset
merged_df = pd.merge(
    fermi_df,
    uci_df,
    on=compound_col,
    how="inner",
    suffixes=("", "_uci_dup"),
)

# Remove any duplicated columns if family/subfamily were present in both files
dup_cols = [col for col in merged_df.columns if col.endswith("_uci_dup")]
merged_df.drop(columns=dup_cols, inplace=True)

# 4. Save the complete dataset ready for training
output_file = "Fermi_169_complete_dataset.csv"
merged_df.to_csv(output_file, index=False)

print(f"Extraction complete!")
print(f"Total compounds matched: {len(merged_df)}")
print(f"Total features/columns: {merged_df.shape[1]}")

Extraction complete!
Total compounds matched: 782
Total features/columns: 87


In [7]:
import pandas as pd

df = pd.read_csv('Fermi_169_complete_dataset.csv')

print(f"Before dedup: {df.shape}")
print(f"Unique materials: {df['material'].nunique()}")

# Keep one row per material — mean Tc, first for everything else
df_169 = df.groupby('material', as_index=False).agg(
    {
        'critical_temp' : 'mean',   # average across duplicate Tc measurements
        'vF (m/s)'      : 'first',  # vF is identical across duplicates
        **{col: 'first' for col in df.columns 
           if col not in ['material', 'critical_temp', 'vF (m/s)']}
    }
)

print(f"\nAfter dedup: {df_169.shape}")
print(f"Unique materials: {df_169['material'].nunique()}")

# Sanity check — vF should have zero variance per material
assert df_169['material'].nunique() == len(df_169), "Duplicates still exist!"
assert df_169['vF (m/s)'].nunique() > 1, "vF looks wrong"

print(f"\nvF range: {df_169['vF (m/s)'].min():,.0f} – {df_169['vF (m/s)'].max():,.0f} m/s")
print(f"Tc range: {df_169['critical_temp'].min():.1f} – {df_169['critical_temp'].max():.1f} K")

df_169.to_csv('Fermi_169_unique_v7.csv', index=False)
print("\nSaved: Fermi_169_unique_v7.csv ✓")

Before dedup: (782, 87)
Unique materials: 162

After dedup: (162, 87)
Unique materials: 162

vF range: 5,000 – 1,800,000 m/s
Tc range: 0.0 – 135.0 K

Saved: Fermi_169_unique_v7.csv ✓


In [8]:
# Check which materials had high Tc variance before averaging
tc_var = df.groupby('material')['critical_temp'].agg(['mean','std','count'])
high_var = tc_var[tc_var['std'] > 10].sort_values('std', ascending=False)
print("\nMaterials with high Tc variance (std > 10K):")
print(high_var)


Materials with high Tc variance (std > 10K):
Empty DataFrame
Columns: [mean, std, count]
Index: []


In [8]:

# =============================================================================
# STEP 5 — Define features and target
# =============================================================================
numeric_features     = [c for c in df_train.columns if c != 'critical_temp']  # all 81
categorical_features = ['family', 'subfamily']   # ← both family AND subfamily

X_train = df_169[numeric_features + categorical_features]
y_train = np.log10(df_169['vF (m/s)'].values)   # log10: 5k–1.8M → 3.7–6.3
groups  = df_169['material'].values                  # for GroupKFold

# print(f"\nX_train shape : {X_train.shape}")
print(f"y_train range : {y_train.min():.3f} – {y_train.max():.3f}  (log10 scale)\n")



y_train range : 3.699 – 6.255  (log10 scale)

